# VinText DBNet + CRNN E2E (12 epochs, separated pipeline)

**Stage 1 (other notebook):** `dbnet-det-vintext-12epochs.ipynb` trains DBNet and saves `dbnet_resnet18_det_vintext_12epochs.pth`.

**This notebook:**
1. Loads the pretrained DBNet and **freezes** it.
2. Trains **CRNN only** on GT word crops (same `VN_BASE_CHARS` / NFD alphabet).
3. Evaluates each epoch and at the end: **Det P/R/H-mean**, **E2E P/R/H-mean**, **CER/WER** on IoU≥0.5 matched pairs (same as Paddle E2E notebook).

**Reproducibility:** `SEED = 42`


In [1]:
!pip install -q pyclipper shapely editdistance


In [2]:
# ============================================================
# Imports, paths, reproducibility (SEED=42)
# ============================================================
import os, glob, re, json, random, unicodedata
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import resnet18, ResNet18_Weights
from tqdm import tqdm

import pyclipper
from shapely.geometry import Polygon
import editdistance

BASE      = "/kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext"
TRAIN_DIR = os.path.join(BASE, "train_images")
VAL_DIR   = os.path.join(BASE, "val_image")
TEST_DIR  = os.path.join(BASE, "test_image")
LABEL_DIR = os.path.join(BASE, "labels")
OUT_DIR   = "/kaggle/working/dbnet_crnn_e2e_vintext"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  SEED: {SEED}")


Device: cuda  |  SEED: 42


In [3]:
# ============================================================
# Vietnamese alphabet (NFD, 106 chars + CTC blank)
# Same as crnn-dbnet-end-to-end(12epochs).ipynb — paper Sec. 5.3
# ============================================================
SPECIAL_BLANK = "<BLANK>"

VN_BASE_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " !\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
    "\u0301"   # acute
    "\u0300"   # grave
    "\u0309"   # hook above
    "\u0323"   # dot below
    "\u0303"   # tilde
    "\u0302"   # circumflex
    "\u0306"   # breve
    "\u031b"   # horn
    "\u0111"   # d with stroke
    "\u0110"   # D with stroke
)

_seen, _char_set = set(), []
for c in VN_BASE_CHARS:
    if c not in _seen:
        _char_set.append(c); _seen.add(c)

CHARS       = [SPECIAL_BLANK] + _char_set
CHAR2IDX    = {c: i for i, c in enumerate(CHARS)}
IDX2CHAR    = {i: c for c, i in CHAR2IDX.items()}
NUM_CLASSES = len(CHARS)
print(f"Alphabet size (incl. CTC blank): {NUM_CLASSES}")


def text_to_indices(text: str) -> List[int]:
    text = unicodedata.normalize("NFD", text.strip())
    return [CHAR2IDX[c] for c in text if c in CHAR2IDX]


def indices_to_text(indices) -> str:
    return "".join(IDX2CHAR.get(i, "") for i in indices if i != 0)


def normalize_for_match(s: str) -> str:
    return unicodedata.normalize("NFD", s).strip()


Alphabet size (incl. CTC blank): 106


In [4]:
# ============================================================
# VinText annotation parser
# ============================================================
def parse_annotation(gt_path: str) -> List[Tuple[np.ndarray, str]]:
    records = []
    if not os.path.exists(gt_path):
        return records
    with open(gt_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",", 8)
            if len(parts) < 9:
                continue
            try:
                coords = list(map(float, parts[:8]))
            except ValueError:
                continue
            transcript = parts[8].strip()
            pts = np.array(coords, dtype=np.float32).reshape(4, 2)
            records.append((pts, transcript))
    return records


def _img_idx_from_name(name: str):
    m = re.search(r"(\d+)", os.path.basename(name))
    return int(m.group(1)) if m else None


def gather_image_records(img_dir: str, label_dir: str):
    for ip in sorted(glob.glob(os.path.join(img_dir, "*.jpg")) +
                     glob.glob(os.path.join(img_dir, "*.png"))):
        idx = _img_idx_from_name(ip)
        gt = os.path.join(label_dir, f"gt_{idx}.txt")
        records = parse_annotation(gt)
        if records:
            yield ip, records


In [5]:
# ============================================================
# 4. DBNet ground-truth generation
#    Implements the prob/threshold/mask map construction from
#    Liao et al., "Real-time Scene Text Detection with
#    Differentiable Binarization" (AAAI'20).
# ============================================================
def shrink_polygon(poly: np.ndarray, shrink_ratio: float = 0.4) -> Optional[np.ndarray]:
    """Vatti-clip shrink, used to build the prob-map GT (text core)."""
    try:
        p = Polygon(poly)
    except Exception:
        return None
    if (not p.is_valid) or p.area <= 1.0:
        return None
    distance = p.area * (1.0 - shrink_ratio ** 2) / max(p.length, 1e-6)
    pco = pyclipper.PyclipperOffset()
    try:
        pco.AddPath(np.round(poly).astype(int).tolist(),
                    pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
        shrunk = pco.Execute(-distance)
    except Exception:
        return None
    if not shrunk:
        return None
    return np.asarray(shrunk[0], dtype=np.int32)


def make_db_targets(polygons: List[np.ndarray],
                    transcripts: List[str],
                    h: int, w: int,
                    shrink_ratio: float = 0.4,
                    thresh_min: float = 0.3,
                    thresh_max: float = 0.7):
    """Build DBNet ground-truth maps for one image.

    Returns
    -------
    gt_prob       : (H, W) float32  -- 1 inside shrunk polygons, 0 elsewhere
    gt_thresh     : (H, W) float32  -- distance-decayed threshold map in [thresh_min, thresh_max]
    prob_mask     : (H, W) float32  -- 0 = ignore (e.g. ### regions), 1 = trainable
    thresh_mask   : (H, W) float32  -- 0/1 mask for the L1 threshold loss (only the boundary band)
    """
    gt_prob     = np.zeros((h, w), dtype=np.float32)
    gt_thresh   = np.zeros((h, w), dtype=np.float32)
    prob_mask   = np.ones ((h, w), dtype=np.float32)
    thresh_mask = np.zeros((h, w), dtype=np.float32)

    for poly, txt in zip(polygons, transcripts):
        if poly is None:
            continue
        ignore = txt in ("###", "***", "")
        if ignore:
            cv2.fillPoly(prob_mask, [poly.astype(np.int32)], 0.0)
            continue

        # ---- prob map: shrunk polygon core ----
        shrunk = shrink_polygon(poly, shrink_ratio)
        if shrunk is None:
            continue
        cv2.fillPoly(gt_prob, [shrunk.astype(np.int32)], 1.0)

        # ---- threshold map: distance band between shrunk and dilated polygon ----
        try:
            p = Polygon(poly)
            distance = p.area * (1.0 - shrink_ratio ** 2) / max(p.length, 1e-6)
        except Exception:
            continue
        pco = pyclipper.PyclipperOffset()
        try:
            pco.AddPath(np.round(poly).astype(int).tolist(),
                        pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
            dilated = pco.Execute(distance)
        except Exception:
            continue
        if not dilated:
            continue
        dilated_arr = np.asarray(dilated[0], dtype=np.int32)
        x_min, y_min = dilated_arr.min(axis=0)
        x_max, y_max = dilated_arr.max(axis=0)
        x_min = max(int(x_min), 0); y_min = max(int(y_min), 0)
        x_max = min(int(x_max), w - 1); y_max = min(int(y_max), h - 1)
        if x_max <= x_min or y_max <= y_min:
            continue
        xs = np.arange(x_min, x_max + 1)
        ys = np.arange(y_min, y_max + 1)
        xv, yv = np.meshgrid(xs, ys)

        # Distance from each pixel in the dilated bbox to the nearest polygon edge.
        n = poly.shape[0]
        dist_map = np.full(xv.shape, np.inf, dtype=np.float32)
        for k in range(n):
            x1, y1 = poly[k]
            x2, y2 = poly[(k + 1) % n]
            dx, dy = x2 - x1, y2 - y1
            seg_len_sq = dx * dx + dy * dy + 1e-12
            t = ((xv - x1) * dx + (yv - y1) * dy) / seg_len_sq
            t = np.clip(t, 0.0, 1.0)
            proj_x = x1 + t * dx
            proj_y = y1 + t * dy
            d = np.sqrt((xv - proj_x) ** 2 + (yv - proj_y) ** 2)
            dist_map = np.minimum(dist_map, d)
        dist_map = 1.0 - np.clip(dist_map / max(distance, 1e-6), 0.0, 1.0)

        # Restrict the threshold-loss band to the dilated polygon only.
        band_mask = np.zeros_like(dist_map, dtype=np.float32)
        cv2.fillPoly(band_mask, [dilated_arr - np.array([[x_min, y_min]])], 1.0)
        dist_map = dist_map * band_mask

        np.maximum(gt_thresh[y_min:y_max + 1, x_min:x_max + 1],
                   dist_map,
                   out=gt_thresh[y_min:y_max + 1, x_min:x_max + 1])
        thresh_mask[y_min:y_max + 1, x_min:x_max + 1] = np.maximum(
            thresh_mask[y_min:y_max + 1, x_min:x_max + 1], band_mask
        )

    gt_thresh = gt_thresh * (thresh_max - thresh_min) + thresh_min
    return gt_prob, gt_thresh, prob_mask, thresh_mask


In [6]:
# ============================================================
# 5. Image preprocessing utilities + helpers shared by both stages
# ============================================================
DET_INPUT_SIZE = 640                              # DBNet input resolution
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def letterbox_image(img: np.ndarray, size: int = DET_INPUT_SIZE):
    """Resize keeping aspect ratio, then pad to (size, size).

    Returns: (padded_image, scale, pad_x, pad_y)
    """
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=resized.dtype)
    pad_x = (size - new_w) // 2
    pad_y = (size - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y


def transform_polygons(polys: List[np.ndarray], scale: float, pad_x: int, pad_y: int) -> List[np.ndarray]:
    out = []
    for p in polys:
        q = p.copy().astype(np.float32)
        q[:, 0] = q[:, 0] * scale + pad_x
        q[:, 1] = q[:, 1] * scale + pad_y
        out.append(q)
    return out


def to_tensor_norm(img_bgr: np.ndarray) -> torch.Tensor:
    """BGR uint8 -> ImageNet-normalised float CHW tensor (used by DBNet)."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return torch.from_numpy(img.transpose(2, 0, 1)).contiguous()


def order_quad(pts: np.ndarray) -> np.ndarray:
    """Reorder 4 points as [TL, TR, BR, BL] so they line up with crop_quad's expectations."""
    pts = np.asarray(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]              # TL
    rect[2] = pts[np.argmax(s)]              # BR
    diff = (pts[:, 1] - pts[:, 0])
    rect[1] = pts[np.argmin(diff)]           # TR
    rect[3] = pts[np.argmax(diff)]           # BL
    return rect


# Recognition crop sizes (kept identical to the recognition-only notebook)
IMG_H = 32
IMG_W_MAX = 512


def crop_quad(image_bgr: np.ndarray, pts: np.ndarray, target_h: int = IMG_H) -> np.ndarray:
    """Perspective-warp a 4-point quadrilateral to a target_h x W strip."""
    pts = pts.astype(np.float32)
    w1 = np.linalg.norm(pts[1] - pts[0]); w2 = np.linalg.norm(pts[2] - pts[3])
    h1 = np.linalg.norm(pts[3] - pts[0]); h2 = np.linalg.norm(pts[2] - pts[1])
    W = max(int((w1 + w2) / 2), 1); H = max(int((h1 + h2) / 2), 1)
    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(pts, dst)
    crop = cv2.warpPerspective(image_bgr, M, (W, H))
    scale = target_h / max(crop.shape[0], 1)
    new_w = max(int(crop.shape[1] * scale), 1)
    return cv2.resize(crop, (new_w, target_h))


def preprocess_crop(crop_bgr: np.ndarray) -> torch.Tensor:
    """RGB -> [-1, 1] CHW tensor (matches the recognition-only CRNN notebook)."""
    img = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    return torch.from_numpy(img.transpose(2, 0, 1))


In [7]:
# ============================================================
# Recognition dataset (GT crops for CRNN)
# ============================================================
class RecognitionDataset(Dataset):
    """GT-cropped word dataset for the recogniser (same role as in the CRNN notebook)."""
    def __init__(self, img_dir: str, label_dir: str,
                 max_label_len: int = 25, augment: bool = False):
        self.augment = augment
        self.samples = []
        skipped = 0
        for ip, recs in gather_image_records(img_dir, label_dir):
            for pts, transcript in recs:
                if transcript in ("###", "***", ""):
                    continue
                idx = text_to_indices(transcript)
                if not idx or len(idx) > max_label_len:
                    skipped += 1
                    continue
                self.samples.append((ip, pts, idx))
        print(f"  [rec] {os.path.basename(img_dir)}: {len(self.samples)} crops ({skipped} skipped)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ip, pts, label = self.samples[idx]
        img = cv2.imread(ip)
        if img is None:
            return torch.zeros(3, IMG_H, IMG_H), [1]
        crop = crop_quad(img, pts)
        if self.augment:
            crop = self._aug(crop)
        t = preprocess_crop(crop)
        if t.shape[2] > IMG_W_MAX:
            t = t[:, :, :IMG_W_MAX]
        return t, label

    @staticmethod
    def _aug(crop):
        a = random.uniform(0.7, 1.3); b = random.randint(-20, 20)
        crop = np.clip(crop.astype(np.float32) * a + b, 0, 255).astype(np.uint8)
        if random.random() < 0.3:
            k = random.choice([3, 5])
            crop = cv2.GaussianBlur(crop, (k, k), 0)
        return crop


def rec_collate(batch):
    """Variable-width padded batch for CTC training (same as the CRNN notebook)."""
    batch = sorted(batch, key=lambda x: -x[0].shape[2])
    imgs, labels = zip(*batch)
    max_w = imgs[0].shape[2]
    padded = [F.pad(img, (0, max_w - img.shape[2]), value=-1.0) for img in imgs]
    images         = torch.stack(padded)
    targets        = torch.tensor([i for s in labels for i in s], dtype=torch.long)
    target_lengths = torch.tensor([len(s) for s in labels], dtype=torch.long)
    input_lengths  = torch.tensor([max(1, img.shape[2] // 4) for img in imgs], dtype=torch.long)
    return images, targets, input_lengths, target_lengths


In [8]:
# ============================================================
# 7. DBNet model: ResNet-18 + FPN + dual heads (prob & threshold)
#    Differentiable binarization (Liao et al., AAAI'20).
# ============================================================
class FPN(nn.Module):
    """Lightweight FPN matched to ResNet-18 stage channels (64,128,256,512)."""
    def __init__(self, in_channels=(64, 128, 256, 512), inner_channels: int = 256):
        super().__init__()
        c2, c3, c4, c5 = in_channels
        self.lat5 = nn.Conv2d(c5, inner_channels, 1)
        self.lat4 = nn.Conv2d(c4, inner_channels, 1)
        self.lat3 = nn.Conv2d(c3, inner_channels, 1)
        self.lat2 = nn.Conv2d(c2, inner_channels, 1)
        self.smooth5 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth4 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth3 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth2 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)

    def forward(self, c2, c3, c4, c5):
        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="bilinear", align_corners=False)
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="bilinear", align_corners=False)
        p5 = self.smooth5(p5); p4 = self.smooth4(p4)
        p3 = self.smooth3(p3); p2 = self.smooth2(p2)
        size = p2.shape[-2:]
        p3 = F.interpolate(p3, size=size, mode="bilinear", align_corners=False)
        p4 = F.interpolate(p4, size=size, mode="bilinear", align_corners=False)
        p5 = F.interpolate(p5, size=size, mode="bilinear", align_corners=False)
        return torch.cat([p2, p3, p4, p5], dim=1)        # (B, inner, H/4, W/4)


class DBNet(nn.Module):
    def __init__(self, k: int = 50, pretrained: bool = True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # /4
        self.layer2 = backbone.layer2   # /8
        self.layer3 = backbone.layer3   # /16
        self.layer4 = backbone.layer4   # /32
        self.fpn = FPN()
        inner = 256

        def _head():
            return nn.Sequential(
                nn.Conv2d(inner, inner // 4, 3, padding=1, bias=False),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, inner // 4, 2, stride=2),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, 1, 2, stride=2),
            )

        self.prob_head   = _head()
        self.thresh_head = _head()
        self.k = k

    def forward(self, x):
        c1 = self.stem(x)
        c2 = self.layer1(c1)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        feat   = self.fpn(c2, c3, c4, c5)
        prob   = torch.sigmoid(self.prob_head(feat))         # (B, 1, H, W)
        thresh = torch.sigmoid(self.thresh_head(feat))       # (B, 1, H, W)
        binary = torch.sigmoid(self.k * (prob - thresh))     # differentiable binarization
        return prob, thresh, binary


In [9]:
# ============================================================
# 9. DBNet post-processing: prob map -> 4-point quad polygons
# ============================================================
@torch.no_grad()
def decode_db_polygons(prob_map: torch.Tensor,
                       prob_thresh: float = 0.3,
                       box_thresh: float = 0.55,
                       min_size: float = 3.0,
                       max_candidates: int = 1000,
                       unclip_ratio: float = 1.5):
    """Convert DBNet probability maps to lists of (quad, score) per image.

    Coordinates are returned in the detector's input space (det_size x det_size).
    """
    results = []
    prob_np = prob_map.detach().squeeze(1).cpu().numpy()      # (B, H, W)
    binary  = (prob_np >= prob_thresh).astype(np.uint8)
    for prob, b in zip(prob_np, binary):
        contours, _ = cv2.findContours(b, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        polys = []
        for cnt in contours[:max_candidates]:
            if cnt.shape[0] < 4:
                continue
            epsilon = 0.005 * cv2.arcLength(cnt, True)
            approx  = cv2.approxPolyDP(cnt, epsilon, True).reshape(-1, 2)
            if approx.shape[0] < 4:
                rect = cv2.minAreaRect(cnt)
                approx = cv2.boxPoints(rect).astype(np.float32)
            mask = np.zeros_like(prob, dtype=np.uint8)
            cv2.fillPoly(mask, [approx.astype(np.int32)], 1)
            if mask.sum() == 0:
                continue
            score = float(prob[mask == 1].mean())
            if score < box_thresh:
                continue
            try:
                p = Polygon(approx.astype(np.float64))
                if (not p.is_valid) or p.area < min_size:
                    continue
                distance = p.area * unclip_ratio / max(p.length, 1e-6)
            except Exception:
                continue
            pco = pyclipper.PyclipperOffset()
            try:
                pco.AddPath(np.round(approx).astype(int).tolist(),
                            pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
                expanded = pco.Execute(distance)
            except Exception:
                continue
            if not expanded:
                continue
            poly = np.asarray(expanded[0], dtype=np.float32)
            if poly.shape[0] < 4:
                continue
            # Reduce to a 4-point quadrilateral so it lines up with crop_quad's expectation.
            rect = cv2.minAreaRect(poly)
            quad = cv2.boxPoints(rect).astype(np.float32)
            quad = order_quad(quad)
            polys.append((quad, score))
        results.append(polys)
    return results


In [10]:
# ============================================================
# 10. CRNN recogniser  (identical to the recognition-only notebook)
# ============================================================
class CRNN(nn.Module):
    """CNN + BiLSTM + CTC (Shi et al., 2015) -- kept verbatim from the original notebook."""
    def __init__(self, num_classes: int, hidden_size: int = 256, n_rnn_layers: int = 2):
        super().__init__()

        def _block(ci, co, pool=None):
            layers = [nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co),
                      nn.LeakyReLU(0.2, inplace=True)]
            if pool == "full":
                layers.append(nn.MaxPool2d(2, 2))
            elif pool == "height":
                layers.append(nn.MaxPool2d((2, 1), (2, 1)))
            return nn.Sequential(*layers)

        self.cnn = nn.Sequential(
            _block(3,   64,  "full"),
            _block(64,  128, "full"),
            _block(128, 256),
            _block(256, 256, "height"),
            _block(256, 512),
            _block(512, 512, "height"),
            _block(512, 512, "height"),
            nn.Conv2d(512, 512, (1, 1)),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.rnn = nn.LSTM(512, hidden_size, num_layers=n_rnn_layers,
                           batch_first=False, bidirectional=True,
                           dropout=0.2 if n_rnn_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        feat = self.cnn(x)
        b, c, h, w = feat.size()
        if h > 1:
            feat = F.adaptive_avg_pool2d(feat, (1, w))
        feat = feat.squeeze(2).permute(2, 0, 1)              # (W, B, C)
        out, _ = self.rnn(feat)
        return F.log_softmax(self.fc(out), dim=2)            # (T, B, C)


def ctc_greedy_decode(log_probs: torch.Tensor) -> List[str]:
    _, idx = log_probs.max(2)
    out = []
    for seq in idx.permute(1, 0):
        seq = seq.tolist(); prev = None; chars = []
        for i in seq:
            if i != prev and i != 0:
                chars.append(i)
            prev = i
        out.append(indices_to_text(chars))
    return out


In [ ]:
# ============================================================
# 10b. Dict-guided recognition components (ported from the VinText study)
# ============================================================
DICT_GUIDED_MAX_LEN = 25
DICT_GUIDED_TRAIN_CANDIDATES = 10
DICT_GUIDED_MAX_EDIT = 1
DICT_GUIDED_TEACH_PROB = 0.5
DICT_GUIDED_REC_LOSS_WEIGHT = 0.05
DICT_GUIDED_INFER_CANDIDATES = 1

DICT_PATH_CANDIDATES = [
    "/kaggle/input/vintext-dict-guided/vn_dictionary.txt",
    "/kaggle/input/dict-guided-vintext/vn_dictionary.txt",
    "/kaggle/working/vn_dictionary.txt",
    r"C:\\Users\\Asus\\Downloads\\Đồ án\\Dict-guided Vintext\\dict-guided\\vn_dictionary.txt",
]


def _find_dict_file() -> Optional[str]:
    for p in DICT_PATH_CANDIDATES:
        if os.path.isfile(p):
            return p
    return None


DICT_FILE = _find_dict_file()
if DICT_FILE is None:
    raise FileNotFoundError(
        "vn_dictionary.txt not found. Add it as a Kaggle dataset or copy it to /kaggle/working/."
    )


def _load_dictionary(path: str) -> List[str]:
    words = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            w = unicodedata.normalize("NFD", line.strip())
            if w:
                words.append(w)
    return sorted(set(words))


class TrieNode:
    def __init__(self):
        self.word = None
        self.children = {}


class Trie:
    def __init__(self, words: List[str]):
        self.root = TrieNode()
        for w in words:
            self.insert(w)

    def insert(self, word: str):
        node = self.root
        for letter in word:
            if letter not in node.children:
                node.children[letter] = TrieNode()
            node = node.children[letter]
        node.word = word

    def search(self, word: str) -> bool:
        node = self.root
        for letter in word:
            if letter not in node.children:
                return False
            node = node.children[letter]
        return node.word == word

    def all_levenshtein(self, word: str, max_cost: int):
        current_row = list(range(len(word) + 1))
        results = []
        for letter, child in self.root.children.items():
            self._search_recursive(child, letter, word, current_row, results, max_cost)
        return results

    def _search_recursive(self, node, letter, word, previous_row, results, max_cost):
        columns = len(word) + 1
        current_row = [previous_row[0] + 1]
        for column in range(1, columns):
            insert_cost = current_row[column - 1] + 1
            delete_cost = previous_row[column] + 1
            replace_cost = previous_row[column - 1] + (0 if word[column - 1] == letter else 1)
            current_row.append(min(insert_cost, delete_cost, replace_cost))

        if current_row[-1] <= max_cost and node.word is not None:
            results.append(node.word)

        if min(current_row) <= max_cost:
            for next_letter, next_node in node.children.items():
                self._search_recursive(next_node, next_letter, word, current_row, results, max_cost)


DICT_WORDS = _load_dictionary(DICT_FILE)
DICT_TRIE = Trie(DICT_WORDS)
print(f"Loaded dictionary: {len(DICT_WORDS)} words from {DICT_FILE}")


class DictGuidedRecognizer(nn.Module):
    """CRNN encoder + attention decoder with dict-guided training objective."""
    def __init__(self, num_classes: int, hidden_size: int = 256, n_rnn_layers: int = 2):
        super().__init__()

        def _block(ci, co, pool=None):
            layers = [nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co),
                      nn.LeakyReLU(0.2, inplace=True)]
            if pool == "full":
                layers.append(nn.MaxPool2d(2, 2))
            elif pool == "height":
                layers.append(nn.MaxPool2d((2, 1), (2, 1)))
            return nn.Sequential(*layers)

        self.cnn = nn.Sequential(
            _block(3,   64,  "full"),
            _block(64,  128, "full"),
            _block(128, 256),
            _block(256, 256, "height"),
            _block(256, 512),
            _block(512, 512, "height"),
            _block(512, 512, "height"),
            nn.Conv2d(512, 512, (1, 1)),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.rnn = nn.LSTM(512, hidden_size, num_layers=n_rnn_layers,
                           batch_first=False, bidirectional=True,
                           dropout=0.2 if n_rnn_layers > 1 else 0.0)

        enc_dim = hidden_size * 2
        self.embedding = nn.Embedding(num_classes, hidden_size)
        self.attn_proj = nn.Linear(hidden_size + enc_dim, hidden_size)
        self.decoder = nn.GRU(hidden_size, hidden_size, batch_first=False)
        self.classifier = nn.Linear(hidden_size, num_classes)
        self.num_classes = num_classes

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.cnn(x)
        b, c, h, w = feat.size()
        if h > 1:
            feat = F.adaptive_avg_pool2d(feat, (1, w))
        feat = feat.squeeze(2).permute(2, 0, 1)
        enc, _ = self.rnn(feat)
        return enc

    def _step(self, prev_tokens: torch.Tensor, hidden: torch.Tensor, enc_out: torch.Tensor):
        emb = self.embedding(prev_tokens)
        query = hidden[-1]
        attn = torch.einsum("tbh,bh->tb", enc_out, query)
        attn = F.softmax(attn.transpose(0, 1), dim=1)
        ctx = torch.einsum("bt,tbh->bh", attn, enc_out)
        dec_in = torch.tanh(self.attn_proj(torch.cat([emb, ctx], dim=1))).unsqueeze(0)
        out, hidden = self.decoder(dec_in, hidden)
        logits = self.classifier(out.squeeze(0))
        return logits, hidden

    def sequence_nll(self, images: torch.Tensor, targets_bt: torch.Tensor, teacher_forcing_prob: float = 0.5):
        enc = self.encode(images)
        bsz = images.size(0)
        steps = targets_bt.size(1)
        hidden = torch.zeros(1, bsz, self.embedding.embedding_dim, device=images.device)
        prev = torch.zeros(bsz, dtype=torch.long, device=images.device)
        losses = []
        for t in range(steps):
            logits, hidden = self._step(prev, hidden, enc)
            tgt = targets_bt[:, t]
            ce = F.cross_entropy(logits, tgt, reduction="none")
            mask = (tgt != 0).float()
            losses.append((ce * mask).sum() / torch.clamp(mask.sum(), min=1.0))
            if random.random() > teacher_forcing_prob:
                prev = tgt
            else:
                prev = logits.argmax(dim=1)
        return torch.stack(losses).mean()

    @torch.no_grad()
    def decode_with_logits(self, images: torch.Tensor, max_len: int = 25):
        self.eval()
        enc = self.encode(images)
        bsz = images.size(0)
        hidden = torch.zeros(1, bsz, self.embedding.embedding_dim, device=images.device)
        prev = torch.zeros(bsz, dtype=torch.long, device=images.device)

        all_log_probs = []
        all_tokens = []
        for _ in range(max_len):
            logits, hidden = self._step(prev, hidden, enc)
            log_probs = F.log_softmax(logits, dim=1)
            prev = log_probs.argmax(dim=1)
            all_log_probs.append(log_probs)
            all_tokens.append(prev)

        step_log_probs = torch.stack(all_log_probs, dim=0)
        pred_tokens = torch.stack(all_tokens, dim=1)
        return pred_tokens, step_log_probs

    @torch.no_grad()
    def predict_texts(self, images: torch.Tensor, dictionary: List[str], trie: Trie,
                      num_candidates: int = 1, max_len: int = 25) -> List[str]:
        pred_tokens, step_log_probs = self.decode_with_logits(images, max_len=max_len)
        pred_texts = []
        for i in range(pred_tokens.size(0)):
            greedy = indices_to_text(pred_tokens[i].tolist())
            nearby = list(set(trie.all_levenshtein(greedy, DICT_GUIDED_MAX_EDIT) + [greedy]))
            if not nearby:
                nearby = [greedy]
            nearby = sorted(nearby, key=lambda w: editdistance.eval(greedy, w))[:max(1, num_candidates)]

            best_word = greedy
            best_loss = float("inf")
            for w in nearby:
                idx = text_to_indices(w)[:max_len]
                idx += [0] * (max_len - len(idx))
                nll = 0.0
                for t, ch in enumerate(idx):
                    nll += float(-step_log_probs[t, i, ch].item())
                if nll < best_loss:
                    best_loss = nll
                    best_word = w
            pred_texts.append(best_word)
        return pred_texts


def build_dict_guided_batch(targets: torch.Tensor,
                            tgt_lens: torch.Tensor,
                            dictionary: List[str],
                            trie: Trie,
                            max_len: int = DICT_GUIDED_MAX_LEN,
                            k: int = DICT_GUIDED_TRAIN_CANDIDATES):
    target_list = targets.detach().cpu().tolist()
    lens_list = tgt_lens.detach().cpu().tolist()

    seqs = []
    offset = 0
    for ln in lens_list:
        seq = target_list[offset:offset + ln]
        seqs.append(seq)
        offset += ln

    cand_batch = []
    score_batch = []

    for seq in seqs:
        gt_text = indices_to_text(seq)
        candidate_words = list(set(trie.all_levenshtein(gt_text, DICT_GUIDED_MAX_EDIT) + [gt_text]))
        ranked = sorted([(w, editdistance.eval(gt_text, w)) for w in candidate_words], key=lambda x: x[1])
        ranked = ranked[:k]
        pad_dist = editdistance.eval(gt_text, "###")
        while len(ranked) < k:
            ranked.append(("###", pad_dist))

        encoded = []
        d_scores = []
        for w, d in ranked:
            idx = text_to_indices(w)[:max_len]
            idx += [0] * (max_len - len(idx))
            encoded.append(idx)
            d_scores.append(1.0 / (float(d) + 0.1))

        cand_batch.append(encoded)
        score_batch.append(d_scores)

    cand = torch.tensor(cand_batch, dtype=torch.long, device=targets.device)
    scores = torch.tensor(score_batch, dtype=torch.float32, device=targets.device)
    cand = cand.permute(1, 0, 2).contiguous()
    return cand, scores


def dict_guided_loss(recognizer: DictGuidedRecognizer,
                     images: torch.Tensor,
                     targets: torch.Tensor,
                     tgt_lens: torch.Tensor,
                     dictionary: List[str],
                     trie: Trie,
                     teach_prob: float = DICT_GUIDED_TEACH_PROB):
    cand, dist_scores = build_dict_guided_batch(targets, tgt_lens, dictionary, trie)
    inv_losses = []
    primary = None

    for i in range(cand.size(0)):
        loss_i = recognizer.sequence_nll(images, cand[i], teacher_forcing_prob=teach_prob)
        inv_losses.append(1.0 / torch.clamp(loss_i.detach(), min=1e-6))
        if i == 0:
            primary = loss_i

    model_dist = F.softmax(torch.stack(inv_losses), dim=0)
    dict_dist = F.softmax(dist_scores.mean(dim=0), dim=0)
    kl = F.kl_div(torch.log(torch.clamp(model_dist, min=1e-8)), dict_dist, reduction="batchmean")
    return primary + kl


In [11]:
# ============================================================
# 11. End-to-end inference pipeline:
#      raw image  ->  DBNet  ->  predicted polygons  ->  recognizer  ->  strings
# ============================================================
class SpotterPipeline:
    def __init__(self, detector: DBNet, recognizer: nn.Module, device: torch.device,
                 det_size: int = DET_INPUT_SIZE,
                 prob_thresh: float = 0.3,
                 box_thresh: float = 0.55,
                 unclip_ratio: float = 1.5,
                 min_quad_size: int = 4):
        self.detector   = detector.to(device)
        self.recognizer = recognizer.to(device)
        self.device       = device
        self.det_size     = det_size
        self.prob_thresh  = prob_thresh
        self.box_thresh   = box_thresh
        self.unclip_ratio = unclip_ratio
        self.min_quad_size = min_quad_size

    def detect(self, image_bgr: np.ndarray):
        """Run only the detector; return predicted quads in *original* image coords."""
        h0, w0 = image_bgr.shape[:2]
        canvas, scale, px, py = letterbox_image(image_bgr, self.det_size)
        tensor = to_tensor_norm(canvas).unsqueeze(0).to(self.device)
        prob, _, _ = self.detector(tensor)
        polys_per_image = decode_db_polygons(
            prob, self.prob_thresh, self.box_thresh,
            unclip_ratio=self.unclip_ratio,
        )
        out = []
        for quad, score in polys_per_image[0]:
            quad_orig = quad.copy()
            quad_orig[:, 0] = (quad_orig[:, 0] - px) / max(scale, 1e-6)
            quad_orig[:, 1] = (quad_orig[:, 1] - py) / max(scale, 1e-6)
            quad_orig[:, 0] = np.clip(quad_orig[:, 0], 0, w0 - 1)
            quad_orig[:, 1] = np.clip(quad_orig[:, 1], 0, h0 - 1)
            out.append((quad_orig.astype(np.float32), score))
        return out

    @torch.no_grad()
    def __call__(self, image_bgr: np.ndarray) -> List[Tuple[np.ndarray, str, float]]:
        self.detector.eval(); self.recognizer.eval()
        det_results = self.detect(image_bgr)
        if not det_results:
            return []
        spotted = []
        for quad, score in det_results:
            try:
                crop = crop_quad(image_bgr, quad.astype(np.float32))
            except Exception:
                continue
            if crop.shape[1] < self.min_quad_size:
                continue
            t = preprocess_crop(crop).unsqueeze(0).to(self.device)
            if t.shape[3] > IMG_W_MAX:
                t = t[:, :, :, :IMG_W_MAX]
            text = self.recognizer.predict_texts(
                t,
                DICT_WORDS,
                DICT_TRIE,
                num_candidates=DICT_GUIDED_INFER_CANDIDATES,
                max_len=DICT_GUIDED_MAX_LEN,
            )[0]
            spotted.append((quad, text, score))
        return spotted


In [12]:
# ============================================================
# End-to-end evaluation: Det + E2E P/R/H-mean, CER/WER (IoU-matched)
# Same protocol as vintext-e2e-paddle-ppocr(12epochs).ipynb
# ============================================================
def polygon_iou(p1: np.ndarray, p2: np.ndarray) -> float:
    try:
        a = Polygon(p1); b = Polygon(p2)
        if not a.is_valid: a = a.buffer(0)
        if not b.is_valid: b = b.buffer(0)
        if a.is_empty or b.is_empty:
            return 0.0
        inter = a.intersection(b).area
        union = a.union(b).area
        return float(inter / union) if union > 0 else 0.0
    except Exception:
        return 0.0


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    h = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, h


def evaluate_endtoend(pipeline: SpotterPipeline,
                      img_dir: str, label_dir: str,
                      iou_thresh: float = 0.5,
                      max_images: Optional[int] = None) -> dict:
    pipeline.detector.eval()
    pipeline.recognizer.eval()
    image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")) +
                         glob.glob(os.path.join(img_dir, "*.png")))
    if max_images:
        image_paths = image_paths[:max_images]

    TP = FP = FN = 0
    det_TP = det_FP = det_FN = 0
    char_edits = char_total = 0
    word_edits = word_total = 0
    rec_matched = 0
    n_images = 0

    for ip in tqdm(image_paths, desc=f"E2E eval [{os.path.basename(img_dir)}]"):
        idx = _img_idx_from_name(ip)
        gt = parse_annotation(os.path.join(label_dir, f"gt_{idx}.txt"))
        if not gt:
            continue
        gt_polys = [g[0] for g in gt]
        gt_texts = [normalize_for_match(g[1]) for g in gt]
        gt_dontcare = [t in ("###", "***", "") for t in gt_texts]

        img = cv2.imread(ip)
        if img is None:
            fn_count = sum(1 for d in gt_dontcare if not d)
            FN += fn_count; det_FN += fn_count
            continue

        preds = pipeline(img)
        n_images += 1
        gt_used = [False] * len(gt_polys)

        for poly, text, score in preds:
            best_iou = 0.0; best_j = -1
            for j, gp in enumerate(gt_polys):
                if gt_used[j]:
                    continue
                iou = polygon_iou(poly, gp)
                if iou > best_iou:
                    best_iou, best_j = iou, j

            iou_matched = (best_j >= 0 and best_iou >= iou_thresh)
            if iou_matched and gt_dontcare[best_j]:
                gt_used[best_j] = True
                continue

            if iou_matched:
                gt_used[best_j] = True
                det_TP += 1
                pred_t = normalize_for_match(text)
                gt_t = gt_texts[best_j]
                char_edits += editdistance.eval(pred_t, gt_t)
                char_total += max(len(gt_t), 1)
                gt_w = gt_t.split() if gt_t.split() else [gt_t]
                pr_w = pred_t.split() if pred_t.split() else [pred_t]
                word_edits += editdistance.eval(pr_w, gt_w)
                word_total += max(len(gt_w), 1)
                rec_matched += 1
                if pred_t == gt_t:
                    TP += 1
                else:
                    FP += 1
            else:
                det_FP += 1; FP += 1

        for j, used in enumerate(gt_used):
            if used or gt_dontcare[j]:
                continue
            det_FN += 1; FN += 1

    P, R, H = _prf(TP, FP, FN)
    Pd, Rd, Hd = _prf(det_TP, det_FP, det_FN)
    cer = char_edits / char_total if char_total else 0.0
    wer = word_edits / word_total if word_total else 0.0
    return dict(
        precision=P, recall=R, hmean=H,
        det_precision=Pd, det_recall=Rd, det_hmean=Hd,
        cer=cer, wer=wer, rec_matched=rec_matched,
        TP=TP, FP=FP, FN=FN,
        det_TP=det_TP, det_FP=det_FP, det_FN=det_FN,
        images=n_images,
    )


def print_e2e_results(tag: str, res: dict):
    print(f"  [{tag}] Detection (IoU>=0.5): P={res['det_precision']:.4f}  R={res['det_recall']:.4f}  H-mean={res['det_hmean']:.4f}")
    print(f"  [{tag}] End-to-end:           P={res['precision']:.4f}  R={res['recall']:.4f}  H-mean={res['hmean']:.4f}")
    print(f"  [{tag}] Recognition (IoU-matched): CER={res['cer']:.4f}  WER={res['wer']:.4f}  (n={res['rec_matched']})")


In [13]:
# ============================================================
# Load frozen pretrained DBNet (.pth from detection notebook)
# ============================================================
DET_WORK_DIR = "/kaggle/working/dbnet_det_vintext"
DET_PTH_CANDIDATES = [
    os.path.join(DET_WORK_DIR, "dbnet_resnet18_det_vintext_12epochs.pth"),
    os.path.join(DET_WORK_DIR, "dbnet_det_best.pth"),
    "/kaggle/input/models/khnhtraan/detect/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth",
    "/kaggle/input/models/dbnet-resnet18-det-vintext/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth",
]


def _find_det_pth():
    for p in DET_PTH_CANDIDATES:
        if os.path.isfile(p):
            return p
    for base in glob.glob("/kaggle/input/*"):
        for root, _, files in os.walk(base):
            for name in ("dbnet_resnet18_det_vintext_12epochs.pth", "dbnet_det_best.pth"):
                if name in files:
                    return os.path.join(root, name)
    return None


DET_PTH = _find_det_pth()
if DET_PTH is None:
    raise FileNotFoundError(
        "Pretrained DBNet .pth not found. Run dbnet-det-vintext-12epochs.ipynb first "
        "and add the output as a Kaggle dataset."
    )
print(f"Loading frozen detector: {DET_PTH}")

detector = DBNet(pretrained=False).to(DEVICE)
try:
    ckpt = torch.load(DET_PTH, map_location=DEVICE, weights_only=False)
except TypeError:
    ckpt = torch.load(DET_PTH, map_location=DEVICE)

state = ckpt.get("detector_state", ckpt)
detector.load_state_dict(state, strict=True)
for p in detector.parameters():
    p.requires_grad = False
detector.eval()
print("Detector frozen (eval, no grad).")


Loading frozen detector: /kaggle/input/models/khnhtraan/detect/pytorch/default/1/dbnet_resnet18_det_vintext_12epochs.pth
Detector frozen (eval, no grad).


In [14]:
# ============================================================
# Crop-level recognition accuracy (exact NFD string match on GT crops)
# ============================================================
@torch.no_grad()
def recognition_crop_accuracy(recognizer: nn.Module, loader,
                              max_batches: Optional[int] = None) -> float:
    recognizer.eval()
    correct = total = 0
    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        crops, targets, in_lens, tgt_lens = [x.to(DEVICE) for x in batch]
        pred_texts = recognizer.predict_texts(
            crops,
            DICT_WORDS,
            DICT_TRIE,
            num_candidates=DICT_GUIDED_INFER_CANDIDATES,
            max_len=DICT_GUIDED_MAX_LEN,
        )
        offset = 0
        for i, tl in enumerate(tgt_lens.cpu().tolist()):
            gt_text = indices_to_text(targets[offset:offset + tl].cpu().tolist())
            offset += tl
            if normalize_for_match(pred_texts[i]) == normalize_for_match(gt_text):
                correct += 1
            total += 1
    return correct / max(total, 1)


In [15]:
# ============================================================
# Recognition training (dict-guided objective; frozen DBNet at inference)
# ============================================================
ACC_MAX_BATCHES_TRAIN = 50   # subset of train loader for per-epoch crop accuracy


def train_recognition(recognizer: DictGuidedRecognizer,
                      train_loader, val_loader,
                      detector: DBNet,
                      num_epochs: int = 12,
                      lr: float = 1e-4,
                      grad_clip: float = 5.0,
                      e2e_eval_fn=None,
                      ckpt_path: Optional[str] = None):
    optimizer = AdamW(recognizer.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    history = dict(
        train_loss=[], val_loss=[], train_acc=[], val_acc=[],
        val_e2e_hmean=[], val_det_hmean=[],
    )
    if ckpt_path is None:
        ckpt_path = os.path.join(OUT_DIR, "dbnet_crnn_e2e_best.pth")

    best_metric = -1.0
    for epoch in range(1, num_epochs + 1):
        recognizer.train()
        detector.eval()
        running = 0.0
        pbar = tqdm(train_loader, desc=f"E{epoch}/{num_epochs} rec train")
        for batch in pbar:
            crops, targets, in_lens, tgt_lens = [x.to(DEVICE, non_blocking=True) for x in batch]
            rec_loss = dict_guided_loss(
                recognizer,
                crops,
                targets,
                tgt_lens,
                DICT_WORDS,
                DICT_TRIE,
                teach_prob=DICT_GUIDED_TEACH_PROB,
            )
            loss = DICT_GUIDED_REC_LOSS_WEIGHT * rec_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(recognizer.parameters(), grad_clip)
            optimizer.step()
            running += loss.item()
            pbar.set_postfix(dict_rec=f"{loss.item():.3f}")
        scheduler.step()
        history["train_loss"].append(running / max(len(train_loader), 1))

        recognizer.eval()
        v_loss = 0.0
        steps = 0
        with torch.no_grad():
            for batch in val_loader:
                crops, targets, in_lens, tgt_lens = [x.to(DEVICE) for x in batch]
                rec_loss = dict_guided_loss(
                    recognizer,
                    crops,
                    targets,
                    tgt_lens,
                    DICT_WORDS,
                    DICT_TRIE,
                    teach_prob=DICT_GUIDED_TEACH_PROB,
                )
                v_loss += (DICT_GUIDED_REC_LOSS_WEIGHT * rec_loss).item()
                steps += 1
        v_loss /= max(steps, 1)
        history["val_loss"].append(v_loss)

        train_acc = recognition_crop_accuracy(
            recognizer, train_loader, max_batches=ACC_MAX_BATCHES_TRAIN,
        )
        val_acc = recognition_crop_accuracy(recognizer, val_loader)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        e2e_h = None
        res = None
        if e2e_eval_fn is not None:
            res = e2e_eval_fn()
            e2e_h = res["hmean"]
            history["val_e2e_hmean"].append(e2e_h)
            history["val_det_hmean"].append(res["det_hmean"])

        track = e2e_h if e2e_h is not None else val_acc
        msg = (f"Epoch {epoch}: train_dict={history['train_loss'][-1]:.4f}  val_dict={v_loss:.4f}  "
               f"| train_acc={train_acc:.4f}  val_acc={val_acc:.4f}")
        if e2e_h is not None:
            msg += f"  | val_e2e_H={e2e_h:.4f}  CER={res['cer']:.4f}"
        print(msg)

        if track > best_metric:
            best_metric = track
            torch.save({
                "epoch": epoch,
                "seed": SEED,
                "framework": "pytorch_dbnet_dictguided_e2e_vintext",
                "det_pth_source": DET_PTH,
                "dict_file": DICT_FILE,
                "detector_state": detector.state_dict(),
                "recognizer_state": recognizer.state_dict(),
                "history": history,
                "best_metric": best_metric,
            }, ckpt_path)
            print(f"    saved best -> {ckpt_path}")

    return history, ckpt_path


In [16]:
# ============================================================
# Build recognition data, train dict-guided recognizer (12 epochs), per-epoch E2E metrics
# ============================================================
print("=== Building recognition datasets (GT crops) ===")
rec_train = RecognitionDataset(TRAIN_DIR, LABEL_DIR, augment=True)
rec_val   = RecognitionDataset(VAL_DIR,   LABEL_DIR, augment=False)

rec_train_loader = DataLoader(rec_train, batch_size=64, shuffle=True,
                              collate_fn=rec_collate, num_workers=2,
                              pin_memory=True, drop_last=True)
rec_val_loader   = DataLoader(rec_val, batch_size=64, shuffle=False,
                              collate_fn=rec_collate, num_workers=2,
                              pin_memory=True)

print(f"Rec batches: train={len(rec_train_loader)}  val={len(rec_val_loader)}")

recognizer = DictGuidedRecognizer(NUM_CLASSES).to(DEVICE)
print(f"Recognizer params: {sum(p.numel() for p in recognizer.parameters()):,}")

PRETRAINED_REC = "/kaggle/input/crnn-best/crnn_best.pth"
if os.path.exists(PRETRAINED_REC):
    try:
        ck = torch.load(PRETRAINED_REC, map_location=DEVICE, weights_only=False)
    except TypeError:
        ck = torch.load(PRETRAINED_REC, map_location=DEVICE)
    state = ck.get("model_state", ck.get("recognizer_state", ck))
    recognizer.load_state_dict(state, strict=False)
    print(f"Warm-started recognizer from {PRETRAINED_REC}")

NUM_EPOCHS = 12
CKPT_BEST = os.path.join(OUT_DIR, "dbnet_crnn_e2e_best.pth")

def _val_e2e_eval():
    pipe = SpotterPipeline(detector, recognizer, DEVICE)
    return evaluate_endtoend(pipe, VAL_DIR, LABEL_DIR)

print("\n=== Training dict-guided recognizer (frozen DBNet, 12 epochs) ===")
history, best_path = train_recognition(
    recognizer, rec_train_loader, rec_val_loader, detector,
    num_epochs=NUM_EPOCHS, e2e_eval_fn=_val_e2e_eval,
    ckpt_path=CKPT_BEST,
)

REC_PTH = os.path.join(OUT_DIR, "dbnet_crnn_e2e_vintext_12epochs.pth")
torch.save(torch.load(best_path, map_location="cpu", weights_only=False), REC_PTH)
print(f"\nExported E2E bundle -> {REC_PTH}")


=== Building recognition datasets (GT crops) ===
  [rec] train_images: 25773 crops (3 skipped)
  [rec] val_image: 7199 crops (2 skipped)
Rec batches: train=402  val=113
Recognizer params: 10,337,130

=== Training CRNN only (frozen DBNet, 12 epochs) ===


E2E eval [val_image]: 100%|██████████| 300/300 [00:45<00:00,  6.53it/s]


Epoch 1: train_ctc=4.3969  val_ctc=3.7874  | train_acc=0.0006  val_acc=0.0000  | val_e2e_H=0.0000  CER=0.9424
    saved best -> /kaggle/working/dbnet_crnn_e2e_vintext/dbnet_crnn_e2e_best.pth


E2E eval [val_image]: 100%|██████████| 300/300 [00:45<00:00,  6.57it/s]


Epoch 2: train_ctc=3.5278  val_ctc=3.3049  | train_acc=0.0131  val_acc=0.0049  | val_e2e_H=0.0018  CER=0.8687
    saved best -> /kaggle/working/dbnet_crnn_e2e_vintext/dbnet_crnn_e2e_best.pth


...

Epoch 10: train_ctc=1.1045  val_ctc=1.4644  | train_acc=0.3144  val_acc=0.2284  | val_e2e_H=0.1565  CER=0.3769
    saved best -> /kaggle/working/dbnet_crnn_e2e_vintext/dbnet_crnn_e2e_best.pth


E2E eval [val_image]: 100%|██████████| 300/300 [00:47<00:00,  6.29it/s]


Epoch 11: train_ctc=1.0260  val_ctc=1.4386  | train_acc=0.3387  val_acc=0.2446  | val_e2e_H=0.1616  CER=0.3680
    saved best -> /kaggle/working/dbnet_crnn_e2e_vintext/dbnet_crnn_e2e_best.pth


E2E eval [val_image]: 100%|██████████| 300/300 [00:48<00:00,  6.17it/s]


Epoch 12: train_ctc=0.9835  val_ctc=1.4273  | train_acc=0.3547  val_acc=0.2516  | val_e2e_H=0.1560  CER=0.3720

Exported E2E bundle -> /kaggle/working/dbnet_crnn_e2e_vintext/dbnet_crnn_e2e_vintext_12epochs.pth


In [17]:
# ============================================================
# Training curves — CTC loss & crop recognition accuracy
# ============================================================
_hist = history if "history" in dir() and history.get("train_loss") else None
if _hist is None:
    _hist = torch.load(best_path, map_location="cpu", weights_only=False).get("history", {})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(_hist["train_loss"]) + 1)

axes[0].plot(ep, _hist["train_loss"], "o-", label="train", linewidth=2, markersize=5)
axes[0].plot(ep, _hist["val_loss"], "o-", label="val", linewidth=2, markersize=5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Dict-guided loss")
axes[0].set_title("Train / Val loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, _hist["train_acc"], "o-", label="train", linewidth=2, markersize=5)
axes[1].plot(ep, _hist["val_acc"], "o-", label="val", linewidth=2, markersize=5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Crop accuracy")
axes[1].set_title("Train / Val accuracy (exact match on GT crops)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(OUT_DIR, "training_curves.png")
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Training curves saved -> {curve_path}")


Training curves saved -> /kaggle/working/dbnet_crnn_e2e_vintext/training_curves.png


In [18]:
# ============================================================
# Final evaluation: val + test (Det P/R/H, E2E P/R/H, CER/WER)
# ============================================================
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
recognizer.load_state_dict(ckpt["recognizer_state"])
pipeline = SpotterPipeline(detector, recognizer, DEVICE)

final_results = {}
for tag, img_dir in [("Validation", VAL_DIR), ("Test", TEST_DIR)]:
    print(f"\n--- {tag} ---")
    res = evaluate_endtoend(pipeline, img_dir, LABEL_DIR, iou_thresh=0.5)
    final_results[tag] = res
    print_e2e_results(tag, res)
    with open(os.path.join(OUT_DIR, f"e2e_metrics_{tag.lower()}.json"), "w") as f:
        json.dump(res, f, indent=2)

print("\n" + "=" * 88)
print(f"  {'Split':<12} {'Det H':>7} {'E2E H':>7} {'CER':>7} {'WER':>7}")
print("  " + "-" * 44)
for tag, r in final_results.items():
    print(f"  {tag:<12} {r['det_hmean']:>7.4f} {r['hmean']:>7.4f} {r['cer']:>7.4f} {r['wer']:>7.4f}")
print("=" * 88)



--- Validation ---


E2E eval [val_image]: 100%|██████████| 300/300 [00:47<00:00,  6.31it/s]


  [Validation] Detection (IoU>=0.5): P=0.8294  R=0.4842  H-mean=0.6115
  [Validation] End-to-end:           P=0.1656  R=0.1578  H-mean=0.1616
  [Validation] Recognition (IoU-matched): CER=0.3680  WER=0.8006  (n=3487)

--- Test ---


E2E eval [test_image]: 100%|██████████| 500/500 [01:21<00:00,  6.16it/s]

  [Test] Detection (IoU>=0.5): P=0.8535  R=0.5301  H-mean=0.6540
  [Test] End-to-end:           P=0.1476  R=0.1633  H-mean=0.1550
  [Test] Recognition (IoU-matched): CER=0.3868  WER=0.8271  (n=5337)

  Split          Det H   E2E H     CER     WER
  --------------------------------------------
  Validation    0.6115  0.1616  0.3680  0.8006
  Test          0.6540  0.1550  0.3868  0.8271


In [19]:
# ============================================================
# Sample E2E inference — GT (green) vs pred boxes + text (cyan)
# ============================================================
from PIL import Image, ImageDraw, ImageFont
import subprocess

ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
recognizer.load_state_dict(ckpt["recognizer_state"])
pipeline = SpotterPipeline(detector, recognizer, DEVICE)

FONT_PATH = "/kaggle/working/NotoSans-Regular.ttf"
if not os.path.exists(FONT_PATH):
    try:
        subprocess.run([
            "wget", "-q", "-O", FONT_PATH,
            "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf",
        ], check=True)
    except Exception as e:
        print(f"(font download skipped: {e})")


def _font(size: int):
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def draw_text_pil(canvas_bgr, text, position, color_rgb, font_size=18, bg_rgb=(0, 0, 0)):
    pil = Image.fromarray(cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)
    font = _font(font_size)
    bbox = draw.textbbox(position, text, font=font)
    pad = 2
    draw.rectangle([bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad], fill=bg_rgb)
    draw.text(position, text, font=font, fill=color_rgb)
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def visualize_e2e(pipe: SpotterPipeline, img_dir: str, label_dir: str,
                    n_samples: int = 6, out_subdir: str = "e2e_inference_samples"):
    out_viz = os.path.join(OUT_DIR, out_subdir)
    os.makedirs(out_viz, exist_ok=True)
    image_paths = sorted(
        glob.glob(os.path.join(img_dir, "*.jpg")) +
        glob.glob(os.path.join(img_dir, "*.png"))
    )[:n_samples]

    for ip in image_paths:
        img = cv2.imread(ip)
        if img is None:
            continue
        idx = _img_idx_from_name(ip)
        gt = parse_annotation(os.path.join(label_dir, f"gt_{idx}.txt"))
        preds = pipe(img)

        canvas = img.copy()
        font_size = max(14, img.shape[0] // 55)
        for poly, txt in gt:
            cv2.polylines(canvas, [poly.astype(np.int32)], True, (0, 200, 0), 2)
            x = int(poly[:, 0].min()); y = int(poly[:, 1].min())
            canvas = draw_text_pil(
                canvas, f"GT: {txt}",
                (x, max(font_size + 4, y - 2 * font_size - 6)),
                color_rgb=(0, 230, 0), font_size=font_size,
            )
        for poly, text, score in preds:
            cv2.polylines(canvas, [poly.astype(np.int32)], True, (255, 200, 0), 2)
            x = int(poly[:, 0].min()); y = int(poly[:, 1].min())
            canvas = draw_text_pil(
                canvas, f"PR: {text} ({score:.2f})",
                (x, max(font_size + 4, y - font_size - 2)),
                color_rgb=(0, 215, 255), font_size=font_size, bg_rgb=(20, 20, 80),
            )

        fig, axes = plt.subplots(1, 2, figsize=(22, 11))
        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original — {os.path.basename(ip)}")
        axes[0].axis("off")
        axes[1].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        axes[1].set_title("GT (green) vs Predicted (cyan)")
        axes[1].axis("off")
        plt.tight_layout()
        save_path = os.path.join(out_viz, f"e2e_{Path(ip).stem}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  saved -> {save_path}")
    print(f"\nE2E sample visualizations -> {out_viz}")


visualize_e2e(pipeline, TEST_DIR, LABEL_DIR, n_samples=6)


  saved -> /kaggle/working/dbnet_crnn_e2e_vintext/e2e_inference_samples/e2e_im1501.png
  saved -> /kaggle/working/dbnet_crnn_e2e_vintext/e2e_inference_samples/e2e_im1502.png
  saved -> /kaggle/working/dbnet_crnn_e2e_vintext/e2e_inference_samples/e2e_im1503.png
  saved -> /kaggle/working/dbnet_crnn_e2e_vintext/e2e_inference_samples/e2e_im1504.png
  saved -> /kaggle/working/dbnet_crnn_e2e_vintext/e2e_inference_samples/e2e_im1505.png
  saved -> /kaggle/working/dbnet_crnn_e2e_vintext/e2e_inference_samples/e2e_im1506.png

E2E sample visualizations -> /kaggle/working/dbnet_crnn_e2e_vintext/e2e_inference_samples
